In [1]:
# %% [Cell 1] 대규모 파생 변수(Feature Engineering) 생성 및 정합성 확보
import pandas as pd
import numpy as np

print("[Step 1] 대규모 파생 변수(Feature Engineering) 생성 시작")

# 1. 절대 경로 데이터 로드 및 엄격한 결측치 정제
file_path = r"C:\dev\project\SKN27-2nd-1TEAM\data\Telco_customer_churn - Telco_Churn.csv"
df = pd.read_csv(file_path)

# [수정] Total Charges 빈칸을 NaN 처리 후, 결측치(Tenure 0인 신규가입자) 행 완전 삭제
df['Total Charges'] = pd.to_numeric(df['Total Charges'].replace(' ', np.nan))
df.dropna(subset=['Total Charges'], inplace=True)

df['Churn Value'] = df['Churn Value'].astype(int)

# 2. 서비스 카운트 기반 변수 (밀집도) - [수정] apply 제거 및 Vectorized 연산으로 속도 최적화
internet_services = ['Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies']
df['Total_Internet_Services'] = (df[internet_services] == 'Yes').sum(axis=1)
df['Total_Streaming'] = (df[['Streaming TV', 'Streaming Movies']] == 'Yes').sum(axis=1)
df['Total_Security'] = (df[['Online Security', 'Online Backup', 'Device Protection', 'Tech Support']] == 'Yes').sum(axis=1)

# 3. 요금(Financial) 기반 연속형 비율 변수
df['Extra_Charges'] = df['Total Charges'] - (df['Monthly Charges'] * df['Tenure Months'])
df['Price_per_Service'] = df['Monthly Charges'] / (df['Total_Internet_Services'] + 1)
df['Total_to_Monthly_Ratio'] = df['Total Charges'] / (df['Monthly Charges'] + 1e-5)
df['Tenure_to_Monthly_Ratio'] = df['Tenure Months'] / (df['Monthly Charges'] + 1e-5)
df['Monthly_to_Median_Ratio'] = df['Monthly Charges'] / df['Monthly Charges'].median()

# 4. 그룹 통계 기반 변수 (집단 내 고객의 요금/유지기간 상대적 위치)
df['Avg_Monthly_by_Contract'] = df.groupby('Contract')['Monthly Charges'].transform('mean')
df['Diff_from_Contract_Monthly'] = df['Monthly Charges'] - df['Avg_Monthly_by_Contract']
df['Avg_Tenure_by_Contract'] = df.groupby('Contract')['Tenure Months'].transform('mean')
df['Diff_from_Contract_Tenure'] = df['Tenure Months'] - df['Avg_Tenure_by_Contract']

# 5. 고객 인구통계 및 관계 기반 이진 변수
df['Is_Full_Family'] = ((df['Partner'] == 'Yes') & (df['Dependents'] == 'Yes')).astype(int)
df['Is_Single_Senior'] = ((df['Senior Citizen'] == 1) & (df['Partner'] == 'No') & (df['Dependents'] == 'No')).astype(int)
df['Is_Independent_Youth'] = ((df['Senior Citizen'] == 0) & (df['Partner'] == 'No') & (df['Dependents'] == 'No')).astype(int)

# 6. 행동 및 가입 특성 기반 지표
df['Is_New_Customer'] = (df['Tenure Months'] <= 6).astype(int)
df['Is_Long_Term_Customer'] = (df['Tenure Months'] >= 60).astype(int)
df['Has_Internet_But_No_Service'] = ((df['Internet Service'] != 'No') & (df['Total_Internet_Services'] == 0)).astype(int)
df['Has_All_Services'] = (df['Total_Internet_Services'] == 6).astype(int)
df['Is_Auto_Payment'] = df['Payment Method'].astype(str).str.contains('automatic', case=False).astype(int)

# 7. 복합 가설 기반 이탈 위험군 (High Risk Flags)
df['Risk_Fiber_MtM'] = ((df['Internet Service'] == 'Fiber optic') & (df['Contract'] == 'Month-to-month')).astype(int)
df['Risk_Payment_Friction'] = ((df['Payment Method'] == 'Electronic check') & (df['Paperless Billing'] == 'Yes')).astype(int)
df['Risk_High_Charge_MtM'] = ((df['Contract'] == 'Month-to-month') & (df['Monthly Charges'] > df['Monthly Charges'].median())).astype(int)
df['Risk_No_TechSupport_Fiber'] = ((df['Internet Service'] == 'Fiber optic') & (df['Tech Support'] == 'No')).astype(int)

# 8. 비선형(다항) 파생 변수
df['Tenure_Sq'] = df['Tenure Months'] ** 2
df['Monthly_Charges_Sq'] = df['Monthly Charges'] ** 2
df['Tenure_x_Monthly'] = df['Tenure Months'] * df['Monthly Charges']

print(f"결과: 대규모 파생 변수 세팅 완료. 최종 데이터 형상: {df.shape}")

[Step 1] 대규모 파생 변수(Feature Engineering) 생성 시작
결과: 대규모 파생 변수 세팅 완료. 최종 데이터 형상: (7032, 60)


In [2]:
# [자동 추가됨] 대규모 전처리가 완료된 DataFrame을 모델링 파일에서 쓰도록 내보냅니다.
import os
csv_path = r'C:\dev\project\SKN27-2nd-1TEAM\data\hwan_preprocessed_data.csv'
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
df.to_csv(csv_path, index=False)
print(f'✅ 전처리 완료 데이터가 성공적으로 저장되었습니다! 저장 경로: {csv_path}')


✅ 전처리 완료 데이터가 성공적으로 저장되었습니다! 저장 경로: C:\dev\project\SKN27-2nd-1TEAM\data\hwan_preprocessed_data.csv
